# Data Collection

In [19]:
import os
import pandas as pd

NFL_YEARS = [str(year) for year in range(2010, 2026)]

DATA_FILEPATH = r"Data"

def load_data(NFL_YEAR):
    path = os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv")
    NFL = pd.read_csv(path)
    return NFL

for year in NFL_YEARS:
    df = load_data(year)
    print(f"{year} loaded — {len(df)} teams")

display(df.tail())

2010 loaded — 32 teams
2011 loaded — 32 teams
2012 loaded — 32 teams
2013 loaded — 32 teams
2014 loaded — 32 teams
2015 loaded — 32 teams
2016 loaded — 32 teams
2017 loaded — 32 teams
2018 loaded — 32 teams
2019 loaded — 32 teams
2020 loaded — 32 teams
2021 loaded — 32 teams
2022 loaded — 32 teams
2023 loaded — 32 teams
2024 loaded — 32 teams
2025 loaded — 32 teams


,Tm,W,L,T,W-L%,PF,PA,PD,MoV,SoS,SRS,OSRS,DSRS
27,New Orleans Saints,6,11,0,0.353,306,383,-77,-4.5,-0.3,-4.9,-5.5,0.6
28,Seattle Seahawks,14,3,0,0.824,483,292,191,11.2,1.6,12.8,6.2,6.7
29,Los Angeles Rams,12,5,0,0.706,518,346,172,10.1,2.4,12.5,8.7,3.9
30,San Francisco 49ers,12,5,0,0.706,437,371,66,3.9,2.1,6.0,3.7,2.3
31,Arizona Cardinals,3,14,0,0.176,355,488,-133,-7.8,3.4,-4.4,-1.0,-3.4


# Logistic Regression Model

In [20]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Column indices
TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

NFL_TEAMS = ['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills',
             'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns',
             'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers',
             'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs',
             'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins',
             'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants',
             'New York Jets', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Francisco 49ers',
             'Seattle Seahawks', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders']
NFL_YEARS = [str(year) for year in range(2010, 2026)]

DATA_FILEPATH = "Data"
SCHEDULE_FILEPATH = "Schedule"
MODEL_FILEPATH = "Models/Logistic Regression"
PREDICTIONS_FILEPATH = "Predictions/Logistic Regression"

In [21]:
def read_data(NFL_YEAR, team_name):
    team_data = []
    with open(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"), "r") as file:
        data = pd.read_csv(file)
        for index, row in data.iterrows():
            if row.iloc[TEAM] == team_name:
                team_data.append([
                    row.iloc[WINS], row.iloc[LOSSES], row.iloc[TIES], row.iloc[WIN_PERCENTAGE],
                    row.iloc[POINTS_FOR], row.iloc[POINTS_AGAINST], row.iloc[POINTS_DIFFERENTIAL],
                    row.iloc[MARGIN_OF_VICTORY], row.iloc[STRENGTH_OF_SCHEDULE],
                    row.iloc[SIMPLE_RATING_SYSTEM], row.iloc[OFFENSIVE_SRS], row.iloc[DEFENSIVE_SRS]
                ])
    return pd.DataFrame(team_data, columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS'])

In [22]:
def train_model(NFL_YEAR):
    all_data = [read_data(NFL_YEAR, team) for team in NFL_TEAMS]
    combined_data = pd.concat([d for d in all_data if d is not None and len(d) > 0], ignore_index=True)

    for col in combined_data.columns:
        combined_data[col] = pd.to_numeric(combined_data[col], errors='coerce')
    combined_data = combined_data.dropna()

    features = combined_data.drop(columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'MoV'])
    target = combined_data['W-L%']

    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

In [23]:
def save_model(NFL_YEAR, model):
    os.makedirs(MODEL_FILEPATH, exist_ok=True)
    with open(os.path.join(MODEL_FILEPATH, f"linear_regression_model_{NFL_YEAR}.pkl"), "wb") as file:
        pickle.dump(model, file)

In [24]:
def make_predictions(NFL_YEAR):
    model_path = os.path.join(MODEL_FILEPATH, f"linear_regression_model_{NFL_YEAR}.pkl")
    with open(model_path, "rb") as file:
        model = pickle.load(file)

    schedule_df = pd.read_csv(os.path.join(SCHEDULE_FILEPATH, f"{NFL_YEAR}.csv"))
    if schedule_df.empty:
        print(f"No schedule found for {NFL_YEAR}")
        return

    team_wins  = {team: 0 for team in NFL_TEAMS}
    team_games = {team: 0 for team in NFL_TEAMS}

    try:
        season_data = pd.read_csv(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"))
        team_stats = {row.iloc[TEAM]: read_data(NFL_YEAR, row.iloc[TEAM]) for _, row in season_data.iterrows()}
        team_stats = {k: v for k, v in team_stats.items() if v is not None and len(v) > 0}
    except FileNotFoundError:
        print(f"Data file for {NFL_YEAR} not found")
        return

    drop_cols = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'MoV']
    for _, row in schedule_df.iterrows():
        winner, loser = row['Winner/tie'], row['Loser/tie']
        if winner not in team_stats or loser not in team_stats:
            continue
        winner_pred = model.predict(team_stats[winner].drop(columns=drop_cols))[0]
        loser_pred  = model.predict(team_stats[loser].drop(columns=drop_cols))[0]
        predicted_winner = winner if winner_pred > loser_pred else loser
        predicted_loser  = loser  if winner_pred > loser_pred else winner
        team_wins[predicted_winner] += 1
        team_games[predicted_winner] += 1
        team_games[predicted_loser]  += 1

    predictions = [
        [team,
         team_stats[team]['W-L%'].iloc[0] if team in team_stats else 0.0,
         team_wins[team] / team_games[team] if team_games[team] > 0 else 0.0]
        for team in NFL_TEAMS
    ]
    predictions_df = pd.DataFrame(predictions, columns=['Team', 'Actual W-L%', 'Predicted W-L%'])
    os.makedirs(PREDICTIONS_FILEPATH, exist_ok=True)
    predictions_df.to_csv(os.path.join(PREDICTIONS_FILEPATH, f"predictions_{NFL_YEAR}.csv"), index=False)
    return predictions_df

In [25]:
for year in NFL_YEARS:
    print(f"Training model for {year}...")
    model = train_model(year)
    save_model(year, model)
    print(f"  Model saved.")
    preds = make_predictions(year)
    print(f"  Predictions made.\n")
    display(preds)  # shows the DataFrame inline in the notebook

Training model for 2010...
  Model saved.
  Predictions made.



,Team,Actual W-L%,Predicted W-L%
0,Arizona Cardinals,0.313,0.1250
1,Atlanta Falcons,0.813,0.9375
2,Baltimore Ravens,0.750,0.6875
3,Buffalo Bills,0.250,0.0000
4,Carolina Panthers,0.125,0.0000
5,Chicago Bears,0.688,0.6250
6,Cincinnati Bengals,0.250,0.1250
7,Cleveland Browns,0.313,0.2500
8,Dallas Cowboys,0.375,0.4375
9,Denver Broncos,0.250,0.0000


Training model for 2011...
  Model saved.
  Predictions made.



,Team,Actual W-L%,Predicted W-L%
0,Arizona Cardinals,0.500,0.3125
1,Atlanta Falcons,0.625,0.6875
2,Baltimore Ravens,0.750,0.8125
3,Buffalo Bills,0.375,0.1250
4,Carolina Panthers,0.375,0.4375
5,Chicago Bears,0.500,0.5000
6,Cincinnati Bengals,0.563,0.6250
7,Cleveland Browns,0.250,0.1250
8,Dallas Cowboys,0.500,0.6875
9,Denver Broncos,0.500,0.2500


Training model for 2012...
  Model saved.
  Predictions made.



...

,Team,Actual W-L%,Predicted W-L%
0,Arizona Cardinals,0.235,0.117647
1,Atlanta Falcons,0.412,0.352941
2,Baltimore Ravens,0.765,1.000000
3,Buffalo Bills,0.647,0.823529
4,Carolina Panthers,0.118,0.000000
5,Chicago Bears,0.412,0.294118
6,Cincinnati Bengals,0.529,0.529412
7,Cleveland Browns,0.647,0.823529
8,Dallas Cowboys,0.706,1.000000
9,Denver Broncos,0.471,0.176471


Training model for 2024...
  Model saved.
  Predictions made.



,Team,Actual W-L%,Predicted W-L%
0,Arizona Cardinals,0.471,0.529412
1,Atlanta Falcons,0.471,0.470588
2,Baltimore Ravens,0.706,0.882353
3,Buffalo Bills,0.765,0.941176
4,Carolina Panthers,0.294,0.000000
5,Chicago Bears,0.294,0.235294
6,Cincinnati Bengals,0.529,0.588235
7,Cleveland Browns,0.176,0.000000
8,Dallas Cowboys,0.412,0.235294
9,Denver Broncos,0.588,0.882353


Training model for 2025...
  Model saved.
  Predictions made.



,Team,Actual W-L%,Predicted W-L%
0,Arizona Cardinals,0.176,0.058824
1,Atlanta Falcons,0.471,0.294118
2,Baltimore Ravens,0.471,0.352941
3,Buffalo Bills,0.706,0.823529
4,Carolina Panthers,0.471,0.588235
5,Chicago Bears,0.647,0.764706
6,Cincinnati Bengals,0.353,0.117647
7,Cleveland Browns,0.294,0.294118
8,Dallas Cowboys,0.441,0.294118
9,Denver Broncos,0.824,0.941176


# Neural Network Model

# Random Forest Model

# Naive Bayes Model